### Cholesterol level prediction:

#### Importing libraries:

In [248]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')


In [249]:
df = pd.read_csv("dataset_2190_cholesterol.csv")

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Define numeric and categorical columns
numeric_cols = ['age','trestbps','thalach','oldpeak','ca']  # **exclude target**
categorical_cols = ['sex','cp','fbs','restecg','exang','slope','thal']
target_col = 'chol'

# Fill missing values
num_imputer = SimpleImputer(strategy='mean')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Encode categorical columns
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])


In [250]:
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


In [251]:
X = df[numeric_cols + categorical_cols + ['num']] if 'num' in df.columns else df[numeric_cols + categorical_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [252]:
# Random Forest
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("RF R2:", r2_score(y_test, y_pred_rf))

# Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
print("DT R2:", r2_score(y_test, y_pred_dt))

# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
print("LR R2:", r2_score(y_test, y_pred_lr))

# SVR
svr_model = SVR()
svr_model.fit(X_train, y_train)
y_pred_svr = svr_model.predict(X_test)
print("SVR R2:", r2_score(y_test, y_pred_svr))


RF R2: 0.08089391815910874
DT R2: -0.6635971816758193
LR R2: 0.12827284971496922
SVR R2: -0.003422223306818406


In [253]:
# Assume Linear Regression is best
best_model = lr_model

joblib.dump(best_model, 'best_cholesterol_model.pkl')
joblib.dump(scaler, 'cholesterol_scaler.pkl')

print("✅ Model and scaler saved successfully!")


✅ Model and scaler saved successfully!


In [254]:
def predict_chol(patient_dict):
    import pandas as pd
    import joblib

    # Load model and scaler
    best_model = joblib.load('best_cholesterol_model.pkl')
    scaler = joblib.load('cholesterol_scaler.pkl')

    # Columns used during training
    trained_columns = X_train.columns.tolist()  # automatically matches training
    numeric_cols = ['age','trestbps','thalach','oldpeak','ca']  # features only

    new_data = pd.DataFrame([patient_dict])

    # Add missing columns with 0
    for col in trained_columns:
        if col not in new_data.columns:
            new_data[col] = 0

    # Reorder columns
    new_data = new_data[trained_columns]

    # Scale numeric features
    new_data[numeric_cols] = scaler.transform(new_data[numeric_cols])

    # Predict
    return best_model.predict(new_data)[0]


In [255]:
patient_info = {
    'age': 60,
    'trestbps': 140,
    'thalach': 160,
    'oldpeak': 1.5,
    'ca': 1,
    'sex': 1,
    'cp': 2,
    'fbs': 0,
    'restecg': 1,
    'exang': 0,
    'slope': 2,
    'thal': 2
}

predicted_chol = predict_chol(patient_info)
print("Predicted Cholesterol Level:", predicted_chol)


Predicted Cholesterol Level: 237.7022560188635
